In [ ]:
import sisl
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from utils._wrappers import timeit
from utils.energies import hamiltonian

### Attempt at calculating electrode DOS using self-energies and built-in 
built-in from https://sisl.readthedocs.io/en/latest/tutorials/tutorial_es_1.html

Fails since `MonkhorstPack` class cannot accept numpy array as hamiltonian. Need to convert the self-energy modulated hamiltonian to `Hamiltonian` class 

In [ ]:
# load data
from pathlib import Path
LENGTH, WIDTH, CENTER = 1, 9, 4
basedir = Path(Path.cwd()).resolve().parent
filedir = basedir / "results" / f"L{LENGTH}_W{WIDTH}_C{CENTER}"
electrode = sisl.io.get_sile(filedir / "electrode.xyz").read_geometry()
ribbon = sisl.io.get_sile(filedir / "ribbon.xyz").read_geometry()
device = sisl.io.get_sile(filedir / "device.xyz").read_geometry()
DATA = np.load(filedir / "calculations.npz")
ENERGIES = DATA["energies"]
lr_idx = DATA["lr_idx"]
lr_idx = (lr_idx[0,...], lr_idx[1, ...])

In [ ]:
from utils.energies import lr_energies
from scipy.sparse import coo_matrix


eta = 1e-5
form = "csc"



H = hamiltonian(ribbon)
H.set_nsc([1,1,1])
Hk = H.Hk(format=form, dtype=complex)
Sk = H.Sk(format=form, dtype=complex)

es = np.zeros_like(ENERGIES)

for ei, E in tqdm(enumerate(ENERGIES), total=len(ENERGIES)):
    En = E + 1j*eta
    SE_pair = lr_energies(electrode, En)
    lidx, ridx = lr_idx
    Hk_with_lr = Hk.copy()
    Hk_with_lr[np.ix_(lidx[0], lidx[0])] += SE_pair[0]
    Hk_with_lr[np.ix_(ridx[0], ridx[0])] += SE_pair[1]

    # Hk_with_lr.shape
    coo = coo_matrix(Hk_with_lr)
    for r, CENTER, v in zip(coo.row, coo.col, coo.data):
        H.H[r,CENTER] = v
    H.set_nsc([1,3,1])

    bz = sisl.MonkhorstPack(H, [1, 50, 1])
    bz_average = bz.apply.average
    es[ei] = bz_average.eigenstate(wrap=lambda x: x.DOS(E))

plt.plot(ENERGIES, es)



In [ ]:
@timeit
def electrode_dos(ham, nk, energies):
    """compute DOS from built-ins
    
    Parameters
    ----------
    ham : Hamiltonian of shape (K, K)
        Hamiltonian in question 
    nk : int
        number of k points
    energies : ndarray of shape (N,)
        list of energies to compute the dos

    Returns
    -------
    dos : ndarray of shape (N,)
        density of state
    bs : ndarray of shape (M, N)
        bandstructure eigenvalues 
    lk : ndarray of shape (M,)
        scaled kpoints to linear values for use in plotting
    """
    # ham = hamiltonian(electrode)
    
    dist = sisl.get_distribution(method="lorentzian", smearing=0.05)
    
    bz = sisl.MonkhorstPack(ham, [1, nk, 1])
    kvecs = [[0,0,0], [0.5, 0, 0]]
    bands = sisl.BandStructure(ham, kvecs, 15, [r"$\Gamma$", "$K$"])
    lk, kticks, knames = bands.lineark(ticks=True)
    bs = bands.apply.array.eigh()
    dos = (bz.apply.average).eigenstate(wrap = lambda x: x.DOS(energies, distribution=dist)) 
    return dos, bs, (lk, kticks, knames)

def plot_dos_band(device,  nk, energies):
    """Plot the device band structure and DOS"""
    H = hamiltonian(device)
    H.set_nsc([1, 3, 1]) # make the y-direction repeating - x direction is used for self-energy
    
    dos, bands, k = electrode_dos(H, nk, energies)
    fig, axes = plt.subplots(1,2, sharey=True)
    fig.suptitle(f"Built-in calc L={LENGTH}, W={WIDTH}, C={CENTER}")
    axes[0].set_title("Band structure")    
    axes[0].plot(k[0], bands)    
    axes[0].set_xlabel("$k$", size=15)
    axes[0].set_xticks(k[1])
    axes[0].set_xticklabels(k[2])
    
    axes[1].set_title("DOS")
    axes[1].plot(dos, energies, label="built-in DOS")
    axes[1].set_xlabel("DOS", size=15)
    
    axes[0].set_ylabel("E", size=15)
    fig.tight_layout()
    for ax in axes:
        ax.grid()
    
    return fig, axes

In [ ]:
# multi_LDOS
fig, ax = plot_dos_band(electrode, 20, ENERGIES)
minmax=2
ax[0].set(ylim=(-minmax, minmax))
